# Digital Forge Reel — Colab Renderer

Render the HTML/CSS/JS scene into a vertical MP4 video directly on Google Colab.

**Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU)

## What this notebook does
1. Unzips the project
2. Runs the one-shot setup script (installs Node 20, fonts, ffmpeg, Playwright Chromium, Python deps)
3. Renders the English + Arabic videos
4. Downloads them to your machine / Google Drive

## Chromium note
On Colab we DO NOT use `apt install chromium-browser` — that's a broken snap stub.
Instead, Playwright downloads its own real Chromium binary (this happens automatically in setup).

## Step 1 — Upload the project zip

Upload `digital-forge-reel.zip` to Colab using the file browser on the left (or mount Drive).

Then run the cell below to unzip it.

In [ ]:
# Unzip the project
%cd /content
!unzip -o digital-forge-reel.zip -d /content/
%cd /content/digital-forge-reel
!ls -la

## Step 2 — Run the one-shot setup script

This installs everything: Node 20, fonts, ffmpeg, Python deps, Playwright Chromium binary.

Takes ~5-8 minutes the first time (mostly the Playwright Chromium download + apt installs).

In [ ]:
!bash setup-colab.sh

## Step 3 — Verify GPU availability (optional)

If you're on a GPU runtime, this shows your GPU. NVENC encoding will be auto-selected.

In [ ]:
!nvidia-smi 2>/dev/null || echo 'No NVIDIA GPU — will use CPU encoder (slower but works)'
!ffmpeg -hide_banner -encoders | grep -E 'h264_nvenc|h264_vaapi|h264_qsv|h264_videotoolbox|libx264' | head -10

## Step 4 — Render the English video

Full quality GPU render. With T4 GPU + NVENC, this takes ~10-15 minutes for a 30s reel.

If you're on CPU runtime, see the cell below for a faster (lower quality) option.

In [ ]:
# Full quality GPU render (Colab T4 GPU runtime)
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/digital-forge-reel-en.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder auto \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --log-level info

In [ ]:
# Fast CPU render (no GPU needed) — half-res capture, ~5 min
# Uncomment to use:
# !node bin/forge-render scenes/digital-forge-reel-en.html \
#     --output output/digital-forge-reel-en-fast.mp4 \
#     --music audio/forge_theme.wav \
#     --scale 0.5 \
#     --fps 15 \
#     --target-fps 60 \
#     --interpolate dup

## Step 5 — Render the Arabic video (Algerian Darija)

In [ ]:
!node bin/forge-render scenes/digital-forge-reel-ar.html \
    --output output/digital-forge-reel-ar.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder auto \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --log-level info

## Step 6 — Verify the videos exist

Check that the MP4 files were created and have the right specs.

In [ ]:
!ls -lh output/*.mp4
!ffprobe -v error -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 output/digital-forge-reel-en.mp4 2>/dev/null || echo 'No video yet'

## Step 7 — Download or save to Drive

In [ ]:
# Option A: Save to Google Drive (persistent across sessions)
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/digital-forge-reel
!cp output/*.mp4 /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null || echo 'No MP4s to copy yet'
!ls -lh /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null

In [ ]:
# Option B: Download directly to your machine
from google.colab import files
import os

for f in os.listdir('output'):
    if f.endswith('.mp4'):
        print(f'Downloading {f}...')
        files.download(f'output/{f}')

## Troubleshooting

**"Chromium not found"** → Run `!bash setup-colab.sh` again. If that fails, run:
```python
!npx playwright install chromium
!npx playwright install-deps chromium
```

**"Command '/usr/bin/chromium-browser' requires the chromium snap to be installed"**
This means Playwright is picking up the snap stub instead of its own binary. Fix:
```python
# Force-remove the snap stub so Playwright's binary is used
!sudo apt remove -y chromium-browser
!npx playwright install chromium
```

**"ffmpeg lacks h264_nvenc"** → Make sure you're on a GPU runtime. Switch via Runtime → Change runtime type → T4 GPU, then re-run setup.

**Video is offset / has black bars** → Fixed in this version. The renderer aligns viewport to stage size exactly.

**Out of memory** → Use `--scale 0.5` to capture at half resolution, or `--fps 15` to halve the frame count.

**Render too slow** → Add `--gpu` flag (requires GPU runtime) and use `--encoder nvenc`. Expect 5-10x speedup.

**Want to modify the scene** → Edit `scenes/digital-forge-reel-en.html` (or `-ar.html`) and re-run the render cell. The HTML is self-contained — no build step.